In [ ]:
import numpy as np
import tensorflow as tf
import random
import os
import pandas as pd
from sklearn.model_selection import train_test_split

SEQ_LEN = 10
TARGET_COL = "label"

CLASSES = ["air", "cinnamon", "rosemary"]
NUM_CLASSES = len(CLASSES)

FEATURE_COLS = [
    "humidity_z",
    "log_ratio",
]

In [ ]:
# %%
def load_dataset(base_dir: str) -> pd.DataFrame:
    dfs = []
    session_id = 0

    for class_folder in os.listdir(base_dir):
        if class_folder not in CLASSES: continue

        folder_path = os.path.join(base_dir, class_folder)

        if not os.path.isdir(folder_path):
            continue

        for file in os.listdir(folder_path):
            if not file.endswith(".csv"):
                continue

            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)

            df["session_id"] = session_id

            df["id"] = df.apply(
                lambda x: f"{session_id}_{int(x['sensor_index'])}_{int(x['fingerprint_index'])}",
                axis=1,
            )

            dfs.append(df)
            session_id += 1

    if not dfs:
        raise ValueError("No CSVW files found.")

    return pd.concat(dfs, ignore_index=True)


data = load_dataset("../data/new-dataset")

# Keep only stable rows
data = data[data["label"].str.endswith("Stable")].copy()

# Convert labels
data["label"] = data["label"].replace({"baselineStable": "air"})
data["label"] = data["label"].str.replace("Stable", "", regex=False)

print("Sessions:", data["session_id"].nunique())

In [ ]:
def compute_baseline(df: pd.DataFrame) -> pd.DataFrame:
    baseline = (
        df[df["label"] == "air"]
        .groupby(["session_id", "sensor_index", "position"])["gas_resistance"]
        .mean()
        .rename("R_base")
        .reset_index()
    )

    df = df.merge(
        baseline,
        on=["session_id", "sensor_index", "position"],
        how="left"
    )

    if df["R_base"].isna().any():
        missing = df[df["R_base"].isna()]["session_id"].unique()
        raise ValueError(f"Missing air baseline in sessions: {missing}")

    return df

data = compute_baseline(data)
data.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

sessions = data["session_id"].unique()

train_sessions, temp_sessions = train_test_split(
    sessions,
    test_size=0.3,
    random_state=42,
    shuffle=True
)

val_sessions, test_sessions = train_test_split(
    temp_sessions,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

train_data = data[data["session_id"].isin(train_sessions)].copy()
val_data   = data[data["session_id"].isin(val_sessions)].copy()
test_data  = data[data["session_id"].isin(test_sessions)].copy()

label_encoder = LabelEncoder()

train_data["label_enc"] = label_encoder.fit_transform(train_data[TARGET_COL])
val_data["label_enc"] = label_encoder.transform(val_data[TARGET_COL])
test_data["label_enc"] = label_encoder.transform(test_data[TARGET_COL])

print("Train sessions:", len(train_sessions))
print("Val sessions  :", len(val_sessions))
print("Test sessions :", len(test_sessions))

In [ ]:
def print_split_classes(name, df):
    classes = sorted(df[TARGET_COL].unique())
    print(f"\n{name} classes ({len(classes)}): {classes}")

print_split_classes("Train", train_data)
print_split_classes("Validation", val_data)
print_split_classes("Test", test_data)

In [ ]:
train_data["log_ratio"] = np.log(
    train_data["gas_resistance"] / train_data["R_base"]
)

val_data["log_ratio"] = np.log(
    val_data["gas_resistance"] / val_data["R_base"]
)

test_data["log_ratio"] = np.log(
    test_data["gas_resistance"] / test_data["R_base"]
)

In [ ]:
env_cols = ["humidity"]

for col in env_cols:
    mean = train_data.groupby("sensor_index")[col].mean()
    std  = train_data.groupby("sensor_index")[col].std()

    train_data[f"{col}_z"] = (
        train_data[col] - train_data["sensor_index"].map(mean)
    ) / train_data["sensor_index"].map(std)

    val_data[f"{col}_z"] = (
        val_data[col] - val_data["sensor_index"].map(mean)
    ) / val_data["sensor_index"].map(std)

    test_data[f"{col}_z"] = (
        test_data[col] - test_data["sensor_index"].map(mean)
    ) / test_data["sensor_index"].map(std)

In [ ]:
def keep_complete(df):
    valid_ids = (
        df.groupby("id")
          .size()
          .loc[lambda x: x == SEQ_LEN]
          .index
    )
    return df[df["id"].isin(valid_ids)]

train_data = keep_complete(train_data)
val_data = keep_complete(val_data)
test_data  = keep_complete(test_data)

print("Train fingerprints:", train_data["id"].nunique())
print("Val fingerprints:", val_data["id"].nunique())
print("Test fingerprints :", test_data["id"].nunique())

In [ ]:
train_data

In [ ]:
import numpy as np

def create_sequences_grouped(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    seq_len: int
):
    X, y = [], []
    grouped = df.groupby("id")
    num_skipped = 0
    for _id, group in grouped:
        group = group.sort_values("position")
        data = group[feature_cols].values

        if len(data) == seq_len:
            X.append(data)
            y.append(int(group[target_col].iloc[0]))
        else:
            num_skipped += 1
    print("Number of sequences skipped: ", num_skipped)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


X_train, y_train = create_sequences_grouped(train_data, FEATURE_COLS, 'label_enc', SEQ_LEN)
X_val, y_val = create_sequences_grouped(val_data, FEATURE_COLS, 'label_enc', SEQ_LEN)
X_test, y_test = create_sequences_grouped(test_data, FEATURE_COLS, 'label_enc', SEQ_LEN)

In [ ]:
from keras.models import Sequential
from keras.layers import Conv1D, Dense, Dropout, Input, GlobalAveragePooling1D

model = Sequential([
    Input(shape=(SEQ_LEN, len(FEATURE_COLS))),

    Conv1D(16, kernel_size=3, activation='relu'),
    Conv1D(16, kernel_size=3, activation='relu'),

    GlobalAveragePooling1D(),

    Dense(32, activation='relu'),
    Dropout(0.2),

    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
from keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[early_stop],
    shuffle=True
)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

overall_acc = accuracy_score(y_test, y_pred)
print("Overall Test Accuracy:", overall_acc)

# Confusion matrix for all classes (force full label range)
labels_full = np.arange(NUM_CLASSES)
cm = confusion_matrix(y_test, y_pred, labels=labels_full)

# per-class accuracy (handle zero samples)
denom = cm.sum(axis=1)
per_class_acc = np.divide(cm.diagonal(), denom,
                          out=np.zeros_like(cm.diagonal(), dtype=float),
                          where=denom!=0)

class_names = label_encoder.inverse_transform(labels_full)

for name, acc, n in zip(class_names, per_class_acc, denom):
    print(f"{name:20s} : {acc:.3f}  (n_test={int(n)})")

print("\nClassification Report (only shows classes present in y_test):")
print(classification_report(y_test, y_pred, labels=np.unique(y_test),
                            target_names=label_encoder.inverse_transform(np.unique(y_test))))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Normalize confusion matrix → percentages per true class
cm_percent = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_percent = np.nan_to_num(cm_percent)  # handle divide-by-zero

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_percent, interpolation='nearest')

# Ticks and labels
ax.set_xticks(np.arange(NUM_CLASSES))
ax.set_yticks(np.arange(NUM_CLASSES))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)

ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion Matrix (Percentages)")

# Annotate each cell
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        val = cm_percent[i, j] * 100
        ax.text(j, i, f"{val:.1f}%", ha="center", va="center")

plt.tight_layout()
plt.colorbar(im, ax=ax)
plt.show()

In [ ]:
def evaluate_csv(file_path: str):
    import pandas as pd
    import numpy as np

    print(f"\nEvaluating: {file_path}")

    # Load
    df = pd.read_csv(file_path)

    # --- Match your preprocessing ---

    # Keep only stable rows
    df = df[df["label"].str.endswith("Stable")].copy()

    # Normalize labels
    df["label"] = df["label"].replace({"baselineStable": "air"})
    df["label"] = df["label"].str.replace("Stable", "", regex=False)

    # Fake session_id (single session)
    df["session_id"] = 0

    # Create ID (same as training)
    df["id"] = df.apply(
        lambda x: f"0_{int(x['sensor_index'])}_{int(x['fingerprint_index'])}",
        axis=1,
    )

    # --- Compute baseline (same logic) ---
    baseline = (
        df[df["label"] == "air"]
        .groupby(["session_id", "sensor_index", "position"])["gas_resistance"]
        .mean()
        .rename("R_base")
        .reset_index()
    )

    df = df.merge(
        baseline,
        on=["session_id", "sensor_index", "position"],
        how="left"
    )

    if df["R_base"].isna().any():
        raise ValueError("Missing air baseline in this CSV")

    # --- Feature engineering ---
    df["log_ratio"] = np.log(df["gas_resistance"] / df["R_base"])

    # Use TRAIN stats (must exist in memory!)
    for col in ["humidity"]:
        df[f"{col}_z"] = (
            df[col] - df["sensor_index"].map(train_data.groupby("sensor_index")[col].mean())
        ) / df["sensor_index"].map(train_data.groupby("sensor_index")[col].std())

    # Encode labels
    df["label_enc"] = label_encoder.transform(df["label"])

    # --- Keep valid sequences ---
    def keep_complete(df):
        valid_ids = (
            df.groupby("id")
              .size()
              .loc[lambda x: x == SEQ_LEN]
              .index
        )
        return df[df["id"].isin(valid_ids)]

    df = keep_complete(df)

    # --- Create sequences ---
    def create_sequences(df):
        X, y = [], []
        for _id, group in df.groupby("id"):
            group = group.sort_values("position")
            data = group[FEATURE_COLS].values

            if len(data) == SEQ_LEN:
                X.append(data)
                y.append(int(group["label_enc"].iloc[0]))

        return np.array(X, dtype=np.float32), np.array(y)

    X, y = create_sequences(df)

    if len(X) == 0:
        print("No valid sequences found.")
        return

    # --- Predict ---
    y_pred = np.argmax(model.predict(X), axis=1)

    acc = (y_pred == y).mean()
    print(f"Accuracy: {acc:.4f}")

    return acc

# evaluate_csv("../data/test/rosemary.csv")

In [ ]:
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {i}")

In [ ]:
import tensorflow as tf
import os

converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Removed converter.optimizations = [tf.lite.Optimize.DEFAULT] to disable quantization for debugging

converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Apply the suggested fix for LSTM models
#converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
#converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
  f.write(tflite_model)

basic_model_size = os.path.getsize("model.tflite")
print("Model is %d bytes" % basic_model_size)


In [ ]:
from ai_edge_litert.compiled_model import CompiledModel
import numpy as np # Import numpy as it's used in the loop

model_lite = CompiledModel.from_file("model.tflite")

signature_index = 0

input_buffers = model_lite.create_input_buffers(signature_index)
output_buffers = model_lite.create_output_buffers(signature_index)

n_correct = 0

for i in range(len(X_test)):
    input_data = np.float32(X_test[i])
    input_buffers[0].write(input_data)

    model_lite.run_by_index(signature_index, input_buffers, output_buffers)
    output_array = output_buffers[0].read(NUM_CLASSES, np.float32)

    pred_class = int(np.argmax(output_array))
    true_class = int(y_test[i])

    if pred_class == true_class:
        n_correct += 1

print("LiteRT accuracy:", n_correct / len(X_test))


In [ ]:
with open("model.tflite", "rb") as f:
    data = f.read()

with open("model.h", "w") as f:
    f.write("const unsigned char model[] = {\n")
    
    for i, byte in enumerate(data):
        f.write(f"0x{byte:02x},")
        if (i + 1) % 12 == 0:
            f.write("\n")
    
    f.write("\n};\n")
    f.write(f"const unsigned int model_len = {len(data)};\n")

import os
print(f"Header file, model.h, is {os.path.getsize('model.h'):,} bytes.")

In [ ]:
env_stats = {}

for col in env_cols:
    means = train_data.groupby("sensor_index")[col].mean()
    stds  = train_data.groupby("sensor_index")[col].std()

    env_stats[col] = {
        "mean": means,
        "std": stds
    }

for col in env_cols:
    means = env_stats[col]["mean"]
    stds  = env_stats[col]["std"]

    print(f"\n// {col} normalization")

    # Mean array
    print(f"static const float {col.upper()}_MEAN[{len(means)}][1] = {{")
    for i, sensor in enumerate(means.index):
        val = means[sensor]
        comma = "," if i < len(means) - 1 else ""
        print(f"    {{{val:.8f}f}}{comma} // Index {sensor}")
    print("};\n")

    # Std array
    print(f"static const float {col.upper()}_STD[{len(stds)}][1] = {{")
    for i, sensor in enumerate(stds.index):
        val = stds[sensor]
        comma = "," if i < len(stds) - 1 else ""
        print(f"    {{{val:.8f}f}}{comma} // Index {sensor}")
    print("};")